# 02 — Fine-tune VideoMAEv2-Small for VSL Recognition\n\n**Model**: VideoMAEv2-Small (distilled, ~22M params)  \n**Dataset**: Multi-VSL (WACV 2025) — 50 classes, frontal view  \n**Training**: Google Colab / Kaggle T4 16GB  \n**Inference target**: Local GPU 4GB VRAM\n\n## Architecture\n```\nVideo (16 frames × 224×224) → VideoMAEv2-Small Encoder (ViT-S) → [CLS] token → FC (50 classes) → Softmax\n```\n\n**Pre-trained on**: Kinetics-400 (video action recognition)  \n**Fine-tune on**: Multi-VSL Vietnamese Sign Language

In [ ]:
# ============================================================\n# CẤU HÌNH TRAINING\n# ============================================================\n\n# Model\nMODEL_NAME = "MCG-NJU/videomae-small-finetuned-kinetics"  # Pre-trained VideoMAE-Small\nNUM_CLASSES = 50\nNUM_FRAMES = 16\nIMAGE_SIZE = 224\n\n# Training hyperparameters\nBATCH_SIZE = 8            # 8 works well on T4 16GB with fp16\nLEARNING_RATE = 5e-4      # Fine-tuning LR\nWEIGHT_DECAY = 0.05       # AdamW weight decay\nEPOCHS = 30               # 30-50 epochs\nWARMUP_EPOCHS = 5         # Linear warmup\nLABEL_SMOOTHING = 0.1     # Label smoothing for regularization\nFP16 = True               # Mixed precision (saves VRAM)\n\n# Data\nVAL_RATIO = 0.2\nNUM_WORKERS = 2           # DataLoader workers\nSEED = 42

## 1. Setup

In [ ]:
import os\nimport sys\n\n# === Detect environment ===\ndef detect_environment():\n    try:\n        import google.colab\n        return "colab"\n    except ImportError:\n        pass\n    if os.path.exists("/kaggle/working"):\n        return "kaggle"\n    return "local"\n\nENV = detect_environment()\nprint(f"🖥️ Environment: {ENV}")\n\n# === Paths ===\nif ENV == "colab":\n    from google.colab import drive\n    drive.mount("/content/drive")\n    BASE_DIR = "/content/drive/MyDrive/vsl-recognition"\nelif ENV == "kaggle":\n    BASE_DIR = "/kaggle/working/vsl-recognition"\nelse:\n    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))\n\nDATA_DIR = os.path.join(BASE_DIR, "data", "multi_vsl")\nMODEL_DIR = os.path.join(BASE_DIR, "models")\nos.makedirs(MODEL_DIR, exist_ok=True)\n\nprint(f"📁 Base: {BASE_DIR}")\nprint(f"📁 Data: {DATA_DIR}")\nprint(f"📁 Models: {MODEL_DIR}")

In [ ]:
# === Install dependencies ===\nif ENV in ("colab", "kaggle"):\n    !pip install -q transformers accelerate decord gdown\n\nimport json\nimport time\nimport random\nimport numpy as np\nfrom pathlib import Path\nfrom collections import Counter\n\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import Dataset, DataLoader\nfrom torch.cuda.amp import autocast, GradScaler\nfrom torchvision.transforms import (\n    Resize, CenterCrop, RandomResizedCrop, ColorJitter, Normalize\n)\n\nfrom transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor\n\nimport matplotlib.pyplot as plt\nfrom sklearn.metrics import confusion_matrix, classification_report\nimport seaborn as sns\n\n# Reproducibility\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\nif torch.cuda.is_available():\n    torch.cuda.manual_seed_all(SEED)\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nprint(f"🔧 Device: {device}")\nif torch.cuda.is_available():\n    print(f"   GPU: {torch.cuda.get_device_name(0)}")\n    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Dataset & DataLoader

In [ ]:
# === Video loading ===\ndef load_video(video_path: str, num_frames: int = 16) -> np.ndarray:\n    \"\"\"Load video and uniformly sample frames.\n    Returns: np.ndarray (num_frames, H, W, 3) uint8.\n    \"\"\"\n    try:\n        from decord import VideoReader, cpu\n        vr = VideoReader(video_path, ctx=cpu(0))\n        total = len(vr)\n        if total >= num_frames:\n            indices = np.linspace(0, total - 1, num_frames, dtype=int)\n        else:\n            indices = np.arange(total)\n            indices = np.concatenate([indices, np.full(num_frames - total, total - 1, dtype=int)])\n        return vr.get_batch(indices).asnumpy()\n    except Exception:\n        import cv2\n        cap = cv2.VideoCapture(video_path)\n        frames = []\n        while cap.isOpened():\n            ret, frame = cap.read()\n            if not ret:\n                break\n            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))\n        cap.release()\n        if not frames:\n            raise ValueError(f"Cannot read: {video_path}")\n        frames = np.array(frames)\n        total = len(frames)\n        if total >= num_frames:\n            indices = np.linspace(0, total - 1, num_frames, dtype=int)\n        else:\n            indices = np.concatenate([np.arange(total), np.full(num_frames - total, total - 1, dtype=int)])\n        return frames[indices]\n\n\nclass VSLVideoDataset(Dataset):\n    \"\"\"Video dataset for Multi-VSL.\"\"\"\n    \n    def __init__(self, video_list: list, num_frames: int = 16, \n                 image_size: int = 224, mode: str = "train"):\n        self.video_list = video_list  # list of {"path": ..., "label": ...}\n        self.num_frames = num_frames\n        self.image_size = image_size\n        self.mode = mode\n        self.normalize = Normalize(\n            mean=[0.485, 0.456, 0.406],\n            std=[0.229, 0.224, 0.225]\n        )\n    \n    def __len__(self):\n        return len(self.video_list)\n    \n    def __getitem__(self, idx):\n        item = self.video_list[idx]\n        video_path = item["path"]\n        label = item["label"]\n        \n        # Load frames\n        frames = load_video(video_path, self.num_frames)  # (T, H, W, 3)\n        \n        # Convert to tensor\n        video = torch.from_numpy(frames).float() / 255.0\n        video = video.permute(0, 3, 1, 2)  # (T, 3, H, W)\n        \n        # Apply transforms\n        T = video.shape[0]\n        transformed = []\n        \n        if self.mode == "train":\n            # Random crop (same params for all frames)\n            i, j, h, w = RandomResizedCrop.get_params(\n                video[0], scale=(0.8, 1.0), ratio=(0.9, 1.1)\n            )\n            jitter = ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1)\n            \n            for t in range(T):\n                frame = video[t][:, i:i+h, j:j+w]\n                frame = Resize((self.image_size, self.image_size), antialias=True)(frame)\n                frame = jitter(frame)\n                frame = self.normalize(frame)\n                transformed.append(frame)\n        else:\n            for t in range(T):\n                frame = Resize(self.image_size + 32, antialias=True)(video[t])\n                frame = CenterCrop(self.image_size)(frame)\n                frame = self.normalize(frame)\n                transformed.append(frame)\n        \n        video_tensor = torch.stack(transformed)  # (T, 3, H, W)\n        return video_tensor, label\n\nprint("✅ Dataset class defined")

In [ ]:
# === Load split from notebook 01 ===\nmeta_dir = Path(DATA_DIR).parent\n\n# Check if split exists\nif (meta_dir / "train.json").exists():\n    with open(meta_dir / "train.json") as f:\n        train_data = json.load(f)\n    with open(meta_dir / "val.json") as f:\n        val_data = json.load(f)\n    with open(meta_dir / "metadata.json") as f:\n        metadata = json.load(f)\n    \n    class_names = metadata["class_names"]\n    print(f"✅ Loaded split from notebook 01:")\nelse:\n    # Create split on-the-fly if not found\n    print("⚠️ Split not found, creating from data directory...")\n    data_path = Path(DATA_DIR)\n    all_classes = sorted([d for d in data_path.iterdir() if d.is_dir()])[:NUM_CLASSES]\n    class_names = [d.name for d in all_classes]\n    class_to_idx = {name: idx for idx, name in enumerate(class_names)}\n    \n    train_data, val_data = [], []\n    for cls_dir in all_classes:\n        videos = sorted([str(f) for f in cls_dir.iterdir() if f.suffix.lower() in ('.avi', '.mp4')])\n        random.shuffle(videos)\n        split = max(1, int(len(videos) * 0.8))\n        for v in videos[:split]:\n            train_data.append({"path": v, "label": class_to_idx[cls_dir.name]})\n        for v in videos[split:]:\n            val_data.append({"path": v, "label": class_to_idx[cls_dir.name]})\n\nprint(f"   Classes: {len(class_names)}")\nprint(f"   Train: {len(train_data)} videos")\nprint(f"   Val: {len(val_data)} videos")\n\n# Create DataLoaders\ntrain_dataset = VSLVideoDataset(train_data, NUM_FRAMES, IMAGE_SIZE, mode="train")\nval_dataset = VSLVideoDataset(val_data, NUM_FRAMES, IMAGE_SIZE, mode="eval")\n\ntrain_loader = DataLoader(\n    train_dataset, batch_size=BATCH_SIZE, shuffle=True,\n    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True\n)\nval_loader = DataLoader(\n    val_dataset, batch_size=BATCH_SIZE, shuffle=False,\n    num_workers=NUM_WORKERS, pin_memory=True\n)\n\nprint(f"\\n   Train batches: {len(train_loader)}")\nprint(f"   Val batches: {len(val_loader)}")

## 3. Load VideoMAEv2-Small Pre-trained Model

In [ ]:
# === Load pre-trained VideoMAE model ===\nprint(f"📥 Loading pre-trained model: {MODEL_NAME}")\nprint(f"   (This downloads ~90MB on first run)\\n")\n\nmodel = VideoMAEForVideoClassification.from_pretrained(\n    MODEL_NAME,\n    num_labels=NUM_CLASSES,\n    ignore_mismatched_sizes=True,  # Replace classification head\n)\nmodel = model.to(device)\n\n# Model info\ntotal_params = sum(p.numel() for p in model.parameters())\ntrainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)\nprint(f"✅ Model loaded!")\nprint(f"   Total params: {total_params / 1e6:.1f}M")\nprint(f"   Trainable params: {trainable_params / 1e6:.1f}M")\nprint(f"   Model size: {total_params * 4 / 1e6:.0f} MB (fp32)")\nprint(f"   Classification head: {NUM_CLASSES} classes")

## 4. Training Loop

In [ ]:
# === Optimizer & Scheduler ===\noptimizer = torch.optim.AdamW(\n    model.parameters(),\n    lr=LEARNING_RATE,\n    weight_decay=WEIGHT_DECAY\n)\n\n# Cosine scheduler with warmup\ntotal_steps = EPOCHS * len(train_loader)\nwarmup_steps = WARMUP_EPOCHS * len(train_loader)\n\ndef lr_lambda(step):\n    if step < warmup_steps:\n        return step / max(1, warmup_steps)\n    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)\n    return 0.5 * (1 + np.cos(np.pi * progress))\n\nscheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)\n\n# Loss with label smoothing\ncriterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)\n\n# Mixed precision scaler\nscaler = GradScaler(enabled=FP16)\n\nprint(f"✅ Training setup:")\nprint(f"   Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")\nprint(f"   Scheduler: Cosine with {WARMUP_EPOCHS} warmup epochs")\nprint(f"   Loss: CrossEntropy + Label Smoothing ({LABEL_SMOOTHING})")\nprint(f"   Mixed Precision: {FP16}")\nprint(f"   Total steps: {total_steps}")

In [ ]:
# === Training & Validation functions ===\n\ndef train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):\n    model.train()\n    total_loss = 0\n    correct = 0\n    total = 0\n    \n    for batch_idx, (videos, labels) in enumerate(loader):\n        videos = videos.to(device)  # (B, T, C, H, W)\n        labels = labels.to(device)\n        \n        optimizer.zero_grad()\n        \n        with autocast(enabled=FP16):\n            outputs = model(pixel_values=videos)\n            loss = criterion(outputs.logits, labels)\n        \n        scaler.scale(loss).backward()\n        scaler.unscale_(optimizer)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        scaler.step(optimizer)\n        scaler.update()\n        scheduler.step()\n        \n        total_loss += loss.item()\n        _, predicted = outputs.logits.max(1)\n        total += labels.size(0)\n        correct += predicted.eq(labels).sum().item()\n        \n        if (batch_idx + 1) % 10 == 0:\n            print(f\"    Batch {batch_idx+1}/{len(loader)} | \"\n                  f\"Loss: {loss.item():.4f} | \"\n                  f\"Acc: {100.*correct/total:.1f}%\", end=\"\\r\")\n    \n    return total_loss / len(loader), 100. * correct / total\n\n\n@torch.no_grad()\ndef evaluate(model, loader, criterion, device):\n    model.eval()\n    total_loss = 0\n    correct = 0\n    total = 0\n    all_preds = []\n    all_labels = []\n    \n    for videos, labels in loader:\n        videos = videos.to(device)\n        labels = labels.to(device)\n        \n        with autocast(enabled=FP16):\n            outputs = model(pixel_values=videos)\n            loss = criterion(outputs.logits, labels)\n        \n        total_loss += loss.item()\n        _, predicted = outputs.logits.max(1)\n        total += labels.size(0)\n        correct += predicted.eq(labels).sum().item()\n        \n        all_preds.extend(predicted.cpu().numpy())\n        all_labels.extend(labels.cpu().numpy())\n    \n    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels\n\nprint("✅ Training functions defined")

In [ ]:
# === TRAINING ===\nprint("=" * 60)\nprint(f"🚀 STARTING TRAINING")\nprint(f"   Model: VideoMAEv2-Small")\nprint(f"   Classes: {NUM_CLASSES} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}")\nprint(f"   Device: {device}")\nprint("=" * 60)\n\nhistory = {\n    "train_loss": [], "train_acc": [],\n    "val_loss": [], "val_acc": [],\n    "lr": []\n}\nbest_val_acc = 0.0\nbest_epoch = 0\n\nfor epoch in range(EPOCHS):\n    start_time = time.time()\n    \n    # Train\n    train_loss, train_acc = train_one_epoch(\n        model, train_loader, criterion, optimizer, scheduler, scaler, device\n    )\n    \n    # Validate\n    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)\n    \n    # Record\n    current_lr = optimizer.param_groups[0]["lr"]\n    history["train_loss"].append(train_loss)\n    history["train_acc"].append(train_acc)\n    history["val_loss"].append(val_loss)\n    history["val_acc"].append(val_acc)\n    history["lr"].append(current_lr)\n    \n    elapsed = time.time() - start_time\n    \n    # Save best model\n    if val_acc > best_val_acc:\n        best_val_acc = val_acc\n        best_epoch = epoch + 1\n        save_path = os.path.join(MODEL_DIR, "videomae_vsl_best")\n        model.save_pretrained(save_path)\n        # Also save class names\n        with open(os.path.join(save_path, "class_names.json"), "w") as f:\n            json.dump(class_names, f, ensure_ascii=False)\n    \n    print(f"  Epoch {epoch+1:02d}/{EPOCHS} | "\n          f"Train: {train_acc:.1f}% (loss {train_loss:.4f}) | "\n          f"Val: {val_acc:.1f}% (loss {val_loss:.4f}) | "\n          f"LR: {current_lr:.6f} | "\n          f"Time: {elapsed:.0f}s"\n          f"{' ⭐ BEST' if val_acc >= best_val_acc else ''}")\n\nprint(f"\\n{'=' * 60}")\nprint(f"✅ Training complete!")\nprint(f"   Best val accuracy: {best_val_acc:.1f}% (epoch {best_epoch})")\nprint(f"   Model saved to: {MODEL_DIR}/videomae_vsl_best")\nprint(f"{'=' * 60}")

## 5. Training Curves & Evaluation

In [ ]:
# === Plot training curves ===\nfig, axes = plt.subplots(1, 3, figsize=(18, 5))\n\n# Loss\naxes[0].plot(history["train_loss"], label="Train Loss", color="blue")\naxes[0].plot(history["val_loss"], label="Val Loss", color="red")\naxes[0].set_xlabel("Epoch")\naxes[0].set_ylabel("Loss")\naxes[0].set_title("Training & Validation Loss")\naxes[0].legend()\naxes[0].grid(True, alpha=0.3)\n\n# Accuracy\naxes[1].plot(history["train_acc"], label="Train Acc", color="blue")\naxes[1].plot(history["val_acc"], label="Val Acc", color="red")\naxes[1].axhline(y=best_val_acc, color="green", linestyle="--", alpha=0.5, label=f"Best: {best_val_acc:.1f}%")\naxes[1].set_xlabel("Epoch")\naxes[1].set_ylabel("Accuracy (%)")\naxes[1].set_title("Training & Validation Accuracy")\naxes[1].legend()\naxes[1].grid(True, alpha=0.3)\n\n# Learning rate\naxes[2].plot(history["lr"], color="green")\naxes[2].set_xlabel("Epoch")\naxes[2].set_ylabel("Learning Rate")\naxes[2].set_title("Learning Rate Schedule (Cosine + Warmup)")\naxes[2].grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.savefig(os.path.join(MODEL_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")\nplt.show()\nprint(f"📊 Training curves saved to {MODEL_DIR}/training_curves.png")

In [ ]:
# === Confusion Matrix & Classification Report ===\nprint("📊 Final evaluation on validation set...\\n")\n\n# Load best model for final evaluation\nbest_model = VideoMAEForVideoClassification.from_pretrained(\n    os.path.join(MODEL_DIR, "videomae_vsl_best"),\n    num_labels=NUM_CLASSES,\n)\nbest_model = best_model.to(device)\n\nval_loss, val_acc, all_preds, all_labels = evaluate(\n    best_model, val_loader, criterion, device\n)\n\nprint(f"Best Model Validation Accuracy: {val_acc:.2f}%\\n")\n\n# Classification report\nprint("📋 Classification Report (top 20 classes):")\nreport = classification_report(\n    all_labels, all_preds, \n    target_names=class_names[:NUM_CLASSES],\n    output_dict=True\n)\nprint(classification_report(\n    all_labels, all_preds,\n    target_names=class_names[:NUM_CLASSES],\n    zero_division=0\n))\n\n# Confusion matrix (show first 20 classes for readability)\nmax_show = min(20, NUM_CLASSES)\ncm = confusion_matrix(all_labels, all_preds)\n\nfig, ax = plt.subplots(figsize=(12, 10))\nsns.heatmap(\n    cm[:max_show, :max_show], \n    annot=True, fmt="d", cmap="Blues",\n    xticklabels=class_names[:max_show],\n    yticklabels=class_names[:max_show],\n    ax=ax\n)\nax.set_xlabel("Predicted")\nax.set_ylabel("True")\nax.set_title(f"Confusion Matrix (first {max_show} classes)")\nplt.xticks(rotation=45, ha="right")\nplt.tight_layout()\nplt.savefig(os.path.join(MODEL_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")\nplt.show()

## 6. Test on Sample Videos\n\nVisualize predictions trên một số video từ validation set.

In [ ]:
import cv2\n\n@torch.no_grad()\ndef predict_and_visualize(model, video_path: str, class_names: list, num_frames=16):\n    \"\"\"Predict a video and show frames with result.\"\"\"\n    model.eval()\n    \n    # Load and preprocess\n    frames = load_video(video_path, num_frames)\n    video = torch.from_numpy(frames).float() / 255.0\n    video = video.permute(0, 3, 1, 2)\n    \n    transformed = []\n    for t in range(num_frames):\n        frame = Resize(IMAGE_SIZE + 32, antialias=True)(video[t])\n        frame = CenterCrop(IMAGE_SIZE)(frame)\n        frame = Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])(frame)\n        transformed.append(frame)\n    \n    input_tensor = torch.stack(transformed).unsqueeze(0).to(device)\n    \n    # Predict\n    outputs = model(pixel_values=input_tensor)\n    probs = torch.softmax(outputs.logits[0], dim=0)\n    top5_probs, top5_idx = torch.topk(probs, 5)\n    \n    # Visualize\n    fig, axes = plt.subplots(2, 8, figsize=(16, 5))\n    fig.suptitle(f"True: {Path(video_path).parent.name} | "\n                 f"Pred: {class_names[top5_idx[0]]} ({top5_probs[0]:.1%})", fontsize=12)\n    \n    for i in range(8):\n        axes[0, i].imshow(frames[i*2])\n        axes[0, i].axis("off")\n        axes[0, i].set_title(f"F{i*2}", fontsize=8)\n    \n    # Top-5 bar chart\n    axes[1, 0].remove()\n    axes[1, 1].remove()\n    axes[1, 2].remove()\n    axes[1, 3].remove()\n    ax_bar = fig.add_subplot(2, 2, 3)\n    colors = ['green' if class_names[idx] == Path(video_path).parent.name else 'steelblue' \n              for idx in top5_idx]\n    ax_bar.barh(\n        [class_names[idx] for idx in top5_idx.flip(0)],\n        top5_probs.flip(0).cpu().numpy(),\n        color=colors[::-1]\n    )\n    ax_bar.set_xlim(0, 1)\n    ax_bar.set_title("Top-5 Predictions")\n    \n    for i in range(4, 8):\n        axes[1, i].axis("off")\n    \n    plt.tight_layout()\n    plt.show()\n\n# Show predictions on random val samples\nprint("🎯 Sample predictions from validation set:\\n")\nsamples = random.sample(val_data, min(5, len(val_data)))\nfor sample in samples:\n    predict_and_visualize(best_model, sample["path"], class_names)\n    print()

## 7. Save Training History\n\nLưu lại history để dùng trong notebook 03.

In [ ]:
# Save training history\nhistory_path = os.path.join(MODEL_DIR, "training_history.json")\nwith open(history_path, "w") as f:\n    json.dump(history, f, indent=2)\n\nprint(f"✅ Training history saved to: {history_path}")\nprint(f"\\n📋 Summary:")\nprint(f"   Best val accuracy: {best_val_acc:.2f}% (epoch {best_epoch})")\nprint(f"   Model saved at: {MODEL_DIR}/videomae_vsl_best/")\nprint(f"\\n🔜 Next: Run notebook 03_inference_and_deploy.ipynb to:")\nprint(f"   - Test inference speed")\nprint(f"   - Export model")\nprint(f"   - Run Streamlit demo")